# Q2 — Spark RDD processing: **master** notebook

Driver on the cluster master (`spark://10.8.0.6:7077`), executors on the worker node(s).
Covers Question 2 **a → d**: parsing with an accumulator, RDD-only analytics,
`groupByKey` vs `reduceByKey`, lineage / jobs-stages-tasks / `cache()`.

**Run order**

| # | cell | what it produces |
|---|---|---|
| 1 | connect | app id, driver UI, worker roster, executor table |
| 2 | snapshot | topic → RDD of JSON lines, partition count and `glom()` sizes |
| 3 | (a) | parse + accumulator of failures, distribution per partition |
| 4 | (b) | top-10 wikis, top-10 idwiki pages, top-15 idwiki comment words |
| 5 | (c) | `reduceByKey` vs `groupByKey` timing + shuffle metrics |
| 6 | (d) | `toDebugString`, jobs/stages/tasks table, `cache()` before/after |
| 7 | save | results JSON for the worker notebook to compare against |

The worker notebook (`q2_spark_worker.ipynb`) re-runs the same pipeline on **one JVM with
2 cores** and checks it produces identical results — the distributed-vs-standalone
comparison the case study asks for.


In [1]:
import json, os, re, socket, sys, time, operator, urllib.request, pathlib

# ------------------------------------------------------------------ config
BOOTSTRAP    = os.environ.get("KAFKA_BOOTSTRAP", "10.8.0.6:29092")
TOPIC        = "wiki.recentchange.raw"
MASTER_URL   = "spark://10.8.0.6:7077"
MASTER_UI    = "http://10.8.0.6:8080"
SHARED_DIR   = pathlib.Path(os.environ.get("Q2_SHARED_DIR", "/tmp/spark-events/q2"))
SNAPSHOT_MIN = 200_000          # Q2 asks for >= 200k, or all collected if fewer

# ------------------------------------------------------------------ schema
FIELDS = ("wiki", "type", "title", "bot", "user", "ts", "len_old", "len_new", "ns", "comment")
WIKI, TYPE, TITLE, BOT, USER, TS, LEN_OLD, LEN_NEW, NS, COMMENT = range(len(FIELDS))

STOPWORDS = set("""
yang dan di ke dari ini itu dengan untuk pada adalah tidak akan juga dalam saya kami kita
dia mereka ada bisa sudah belum karena agar oleh sebagai atau jika maka saat setelah sebelum
lebih paling sangat hanya saja para semua beberapa satu dua tiga empat lima hal cara orang
tahun baru lama masih pernah sedang telah harus ingin mau tapi tetapi namun sehingga bahwa
antara tanpa hingga sampai ketika sebab kalau pun lah kah nya ku mu se secara kepada terhadap
merupakan menjadi dibuat diubah ditambah dihapus termasuk berikut yaitu yakni ini itu sini sana
""".split())

# ------------------------------------------------------------------ helpers
def ui_json(sc, path, timeout=20):
    """Spark driver UI REST API (the same numbers the Spark UI shows)."""
    base = sc.uiWebUrl.rstrip("/") + "/api/v1/applications/" + sc.applicationId
    with urllib.request.urlopen(base + path, timeout=timeout) as fh:
        return json.load(fh)


def http_json(url, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as fh:
        return json.load(fh)


_SEEN = {"stages": set(), "jobs": set()}
RECORDS = []          # every measured action ends up here -> the (d) tables


def measure(sc, label, fn, note=""):
    """Run one RDD action and record jobs / stages / tasks / shuffle / seconds.

    The action is wrapped in a Spark job group, so the jobs it submitted can be pulled
    back by name (`statusTracker().getJobIdsForGroup`) instead of guessed at — the
    shuffle bytes then come from the stages of exactly those jobs, as the Spark UI shows.
    """
    tag = f"q2::{label}"
    try:
        sc.setJobGroup(tag, label, False)
    except TypeError:
        sc.setJobGroup(tag, label)

    t0 = time.perf_counter()
    result = fn()
    seconds = time.perf_counter() - t0

    try:
        job_ids = list(sc.statusTracker().getJobIdsForGroup(tag))
    except Exception:                            # noqa: BLE001
        job_ids = []

    stages = jobs = []
    try:
        jobs = ui_json(sc, "/jobs")
        stages = {s["stageId"]: s for s in ui_json(sc, "/stages")}
    except Exception:                            # UI unreachable -> keep the timing only
        pass

    mine = [j for j in jobs if j["jobId"] in job_ids]
    if not mine:                                 # fallback: diff against what we have seen
        mine = [j for j in jobs if j["jobId"] not in _SEEN["jobs"]]
    _SEEN["jobs"].update(j["jobId"] for j in mine)

    stage_ids = sorted({sid for j in mine for sid in j.get("stageIds", [])})
    sel = [stages[sid] for sid in stage_ids if sid in stages]

    rec = {
        "label": label, "note": note, "seconds": round(seconds, 3),
        "jobs": len(mine), "job_ids": sorted(j["jobId"] for j in mine),
        "stages": len(sel), "stage_ids": stage_ids,
        "tasks": sum(s.get("numTasks", 0) for s in sel),
        "shuffle_write": sum(s.get("shuffleWriteBytes", 0) for s in sel),
        "shuffle_read": sum(s.get("shuffleReadBytes", 0) for s in sel),
    }
    RECORDS.append(rec)
    return result


def table(rows, cols, title=""):
    """Fixed-width text table for the report."""
    rows = list(rows)
    w = {c: max(len(c), 6, *(len(str(r.get(c, ""))) for r in rows)) if rows else len(c)
         for c in cols}
    out = []
    if title:
        out.append(title)
    out.append("  ".join(f"{c:>{w[c]}}" for c in cols))
    out.append("-" * (sum(w.values()) + 2 * (len(cols) - 1)))
    for r in rows:
        out.append("  ".join(f"{str(r.get(c, '')):>{w[c]}}" for c in cols))
    text = chr(10).join(out)
    print(text)
    return text


def parse_line(line, fail_acc, ok_acc=None):
    """JSON line -> 10-field tuple; malformed records are counted, never dropped silently.

    `ok_acc` counts *parsed* records, which is how the notebook proves that an uncached
    RDD is recomputed on every action (and stops being recomputed after cache()).
    """
    try:
        d = json.loads(line)
        length = d.get("length") or {}
        rec = (d.get("wiki"), d.get("type"), d.get("title"), bool(d.get("bot")),
               d.get("user"), d.get("timestamp"), length.get("old"), length.get("new"),
               d.get("namespace"), d.get("comment") or "")
        if not rec[WIKI] or not rec[TYPE]:
            raise ValueError("record without wiki/type")
        if ok_acc is not None:
            ok_acc.add(1)
        return rec
    except Exception:                            # noqa: BLE001
        fail_acc.add(1)
        return None


WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)


def words_of(comment):
    """Lower-case alphabetic words, stop words removed."""
    return [w for w in (m.group(0).lower() for m in WORD_RE.finditer(comment or ""))
            if len(w) > 2 and w not in STOPWORDS]


def fmt_bytes(n):
    n = float(n or 0)
    for unit in ("B", "KiB", "MiB", "GiB"):
        if n < 1024 or unit == "GiB":
            return f"{n:.1f} {unit}"
        n /= 1024.0


def kafka_snapshot(spark):
    """Batch read of the whole topic: every executor reads its own Kafka partitions."""
    df = (spark.read.format("kafka")
          .option("kafka.bootstrap.servers", BOOTSTRAP)
          .option("subscribe", TOPIC)
          .option("startingOffsets", "earliest")
          .option("endingOffsets", "latest")
          .load())
    return df


In [2]:
import json, os, socket, time
from pyspark.sql import SparkSession

# PYSPARK_SUBMIT_ARGS in the container already carries
#   --master spark://10.8.0.6:7077 --conf spark.driver.host=10.8.0.6
#   --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3
# so no .master() call here: the kernel *is* the cluster driver.
spark = (SparkSession.builder
         .appName("q2-rdd-master")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("spark version   :", spark.version)
print("sc.master       :", sc.master)
print("application id  :", sc.applicationId)
print("driver UI       :", sc.uiWebUrl)
print("driver hostname :", socket.gethostname())
print("default parallelism:", sc.defaultParallelism)

roster = [w for w in http_json(MASTER_UI + "/json/")["workers"]]
table([{"host": w["host"], "port": w["port"], "cores": w["cores"],
        "memory_MB": w["memory"], "state": w["state"], "ui": w["webuiaddress"]}
       for w in roster], ["host", "port", "cores", "memory_MB", "state", "ui"],
      title="workers registered with the master")
print()

ex = ui_json(sc, "/executors")
table([{"executor": e["id"], "host:port": e["hostPort"], "cores": e["totalCores"],
        "tasks": e["totalTasks"], "input": fmt_bytes(e.get("totalInputBytes"))}
       for e in ex], ["executor", "host:port", "cores", "tasks", "input"],
      title="executors of this application (driver included)")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/spark-ivy/cache
The jars for the packages stored in: /tmp/spark-ivy/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-275187ec-f0f2-4bf7-baeb-08660558144a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.3 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 i

spark version   : 4.1.3
sc.master       : spark://10.8.0.6:7077
application id  : app-20261116140628-0011
driver UI       : http://10.8.0.6:4040
driver hostname : jupyter
default parallelism: 4
workers registered with the master
    host    port   cores  memory_MB   state                    ui
-----------------------------------------------------------------
10.8.0.5   43973       2       2048   ALIVE  http://10.8.0.5:8081
 0.0.0.0   36211       2       2048   ALIVE   http://0.0.0.0:8081
10.8.0.3   39440       2       2048   ALIVE  http://10.8.0.3:8081
10.8.0.6   36105       2       2048   ALIVE  http://10.8.0.6:8081

executors of this application (driver included)
executor       host:port   cores   tasks   input
------------------------------------------------
  driver  10.8.0.6:35031       0       0   0.0 B
       2  10.8.0.6:37941       2       0   0.0 B
       1  10.8.0.3:38342       2       0   0.0 B


'executors of this application (driver included)\nexecutor       host:port   cores   tasks   input\n------------------------------------------------\n  driver  10.8.0.6:35031       0       0   0.0 B\n       2  10.8.0.6:37941       2       0   0.0 B\n       1  10.8.0.3:38342       2       0   0.0 B'

## Step 2 — snapshot of the topic

`spark.read.format("kafka")` in **batch** mode (`startingOffsets=earliest`,
`endingOffsets=latest`): the connector hands each executor the Kafka partitions it owns,
so the read itself is already distributed — no shared directory needed (the workers are
separate machines, so `file://` would not be visible to them).

The RDD below is a list of raw JSON strings, one per Kafka message: this is the input the
question wants parsed "line by line".


In [3]:
kafka_df = kafka_snapshot(spark)
raw_lines = kafka_df.select("value").rdd.map(lambda r: r[0].decode("utf-8"))

n_partitions = raw_lines.getNumPartitions()
print("kafka partitions -> RDD partitions:", n_partitions)
print("topic                      :", TOPIC)
print("broker                     :", BOOTSTRAP)

total = measure(sc, "snapshot count (whole topic)", lambda: raw_lines.count())
print("messages in snapshot       :", total)
if total < SNAPSHOT_MIN:
    print(f"NOTE: {total:,} < {SNAPSHOT_MIN:,} requested — the question allows "
          f"'all messages collected if fewer', so this snapshot is the full capture.")
else:
    print(f"OK: {total:,} >= {SNAPSHOT_MIN:,}")

sizes = raw_lines.glom().map(len).collect()
table([{"partition": i, "records": n} for i, n in enumerate(sizes)],
      ["partition", "records"], title="records per RDD partition (input skew)")

kafka partitions -> RDD partitions: 6
topic                      : wiki.recentchange.raw
broker                     : 10.8.0.6:29092
messages in snapshot       : 57778
NOTE: 57,778 < 200,000 requested — the question allows 'all messages collected if fewer', so this snapshot is the full capture.
records per RDD partition (input skew)
partition  records
------------------
        0     2282
        1     3766
        2    17327
        3     2135
        4    29899
        5     2369


'records per RDD partition (input skew)\npartition  records\n------------------\n        0     2282\n        1     3766\n        2    17327\n        3     2135\n        4    29899\n        5     2369'

## (a) Parse into tuples, count failures with an accumulator, show the distribution

`parse_line()` returns a 10-field tuple and calls `acc.add(1)` on anything malformed
(truncated JSON, empty `wiki`, …) — nothing is dropped silently. `glom()` then materialises
each partition so the element distribution can be inspected directly, and
`mapPartitionsWithIndex` adds the **hostname of the executor** that processed it, which is
the evidence that the work really was distributed.

Note the partition sizes: the six Kafka partitions are very uneven (roughly 2.1k – 29.9k
records). That is not a Spark problem — Q1 keyed the topic on `wiki`, and `commonswiki` /
`wikidatawiki` dominate the stream, so the skew that Q1a predicted shows up here as
task imbalance (see (d) for what that costs).


In [4]:
parse_failures = sc.accumulator(0)   # PySpark takes no name argument
records_seen  = sc.accumulator(0)

parsed = (raw_lines
          .map(lambda line: parse_line(line, parse_failures, records_seen))
          .filter(lambda rec: rec is not None))

n_ok = measure(sc, "(a) parse + count", lambda: parsed.count())
print(f"parsed records        : {n_ok:,}")
print(f"parse failures (acc.) : {parse_failures.value}")
print(f"records seen (acc.)   : {records_seen.value:,}  "
      f"(records parsed so far — grows on every recomputation)")

sizes = parsed.glom().map(len).collect()
table([{"partition": i, "records": n} for i, n in enumerate(sizes)],
      ["partition", "records"], title="(a) elements per partition via glom()")
print()

hosts = (parsed.mapPartitionsWithIndex(
            lambda i, it: [(i, socket.gethostname(), sum(1 for _ in it))])
            .collect())
table([{"partition": p, "executor host": h, "records": n} for p, h, n in hosts],
      ["partition", "executor host", "records"],
      title="(a) which executor host ran which partition")

parsed records        : 57,778
parse failures (acc.) : 0
records seen (acc.)   : 57,778  (records parsed so far — grows on every recomputation)
(a) elements per partition via glom()
partition  records
------------------
        0     2282
        1     3766
        2    17327
        3     2135
        4    29899
        5     2369

(a) which executor host ran which partition
partition  executor host  records
---------------------------------
        0   spark-worker     2282
        1   spark-worker     3766
        2   spark-worker    17327
        3   spark-worker     2135
        4   spark-worker    29899
        5   spark-worker     2369


'(a) which executor host ran which partition\npartition  executor host  records\n---------------------------------\n        0   spark-worker     2282\n        1   spark-worker     3766\n        2   spark-worker    17327\n        3   spark-worker     2135\n        4   spark-worker    29899\n        5   spark-worker     2369'

## (b) Three RDD-only questions

Only `filter`, `map`, `flatMap`, `reduceByKey` and `takeOrdered` are used — no DataFrame API.

| # | question | narrow (N) / wide (W) |
|---|---|---|
| i | top 10 wikis by human edits (`type=edit`, `bot=false`) | filter/map N, **reduceByKey W**, takeOrdered N |
| ii | top 10 most edited pages on `idwiki` | filter/map N, **reduceByKey W**, takeOrdered N |
| iii | top 15 words in `idwiki` comments (stop words removed) | filter/flatMap N, **reduceByKey W**, takeOrdered N |

A wide transformation is the one that needs a shuffle (`reduceByKey`, `groupByKey`,
`sortByKey`, `join`, `repartition`); narrow ones move data within a partition
(`filter`, `map`, `flatMap`, `mapPartitions`).


In [5]:
add = operator.add

def top_wikis_by_human_edits(rdd):
    return (rdd.filter(lambda r: r[TYPE] == "edit" and r[BOT] is False)
               .map(lambda r: (r[WIKI], 1))
               .reduceByKey(add)
               .takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

def top_pages_idwiki(rdd):
    return (rdd.filter(lambda r: r[WIKI] == "idwiki" and r[TYPE] == "edit")
               .map(lambda r: (r[TITLE], 1))
               .reduceByKey(add)
               .takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

def top_words_idwiki_comments(rdd):
    return (rdd.filter(lambda r: r[WIKI] == "idwiki")
               .flatMap(lambda r: [(w, 1) for w in words_of(r[COMMENT])])
               .reduceByKey(add)
               .takeOrdered(15, key=lambda kv: (-kv[1], kv[0])))

b1 = measure(sc, "(b-i) top wikis, human edits", lambda: top_wikis_by_human_edits(parsed))
b2 = measure(sc, "(b-ii) top pages idwiki", lambda: top_pages_idwiki(parsed))
b3 = measure(sc, "(b-iii) top words idwiki comments", lambda: top_words_idwiki_comments(parsed))

table([{"rank": i + 1, "wiki": w, "human_edits": c} for i, (w, c) in enumerate(b1)],
      ["rank", "wiki", "human_edits"], title="(b-i) TOP 10 wikis by human edits")
print()
table([{"rank": i + 1, "page": t, "edits": c} for i, (t, c) in enumerate(b2)],
      ["rank", "page", "edits"], title="(b-ii) TOP 10 most edited pages on idwiki")
print()
table([{"rank": i + 1, "word": w, "count": c} for i, (w, c) in enumerate(b3)],
      ["rank", "word", "count"], title="(b-iii) TOP 15 words in idwiki comments")

(b-i) TOP 10 wikis by human edits
  rank           wiki  human_edits
----------------------------------
     1   wikidatawiki        11211
     2    commonswiki         3529
     3         enwiki         1845
     4         dewiki          476
     5         frwiki          374
     6         ruwiki          254
     7         zhwiki          247
     8         jawiki          224
     9         itwiki          187
    10  yuewiktionary          175

(b-ii) TOP 10 most edited pages on idwiki
  rank                                        page   edits
----------------------------------------------------------
     1             Daftar produk rokok dari Djarum       2
     2                                   Decorient       2
     3                               Eddy Soeparno       2
     4                           Keluarga Punaluan       2
     5                                         Max       2
     6  Pengguna:XmOffr1c!al77/Berkas:Yogi Supardi       2
     7                         

'(b-iii) TOP 15 words in idwiki comments\n  rank           word   count\n-----------------------------\n     1        halaman     267\n     2       kategori     140\n     3           lain     104\n     4        special     104\n     5        bersama     102\n     6     dimasukkan     102\n     7  whatlinkshere     102\n     8    ditambahkan      64\n     9          evans      51\n    10       jonathan      40\n    11           sama      40\n    12          chris      36\n    13       donnelly      22\n    14           rory      22\n    15          davis      20'

## (c) `groupByKey().mapValues(sum)` vs `reduceByKey`

Both variants compute exactly the same result; the difference is **where the reduction
happens**. `reduceByKey` combines per partition before the shuffle (map-side combine), so
the shuffle carries one record per distinct key per partition. `groupByKey` ships every
*value* (all 1s) and only then sums them — much more shuffle traffic and higher peak memory.

The timings below are read from the Spark REST API of this driver's UI (Shuffle Write/Read
bytes per stage), because that is the number the Spark UI shows.


In [6]:
human = (parsed.filter(lambda r: r[TYPE] == "edit" and r[BOT] is False)
               .map(lambda r: (r[WIKI], 1)))

# Same input for both variants: persist it once so the comparison measures the
# aggregation, not the re-read of Kafka.
human_p = human.persist()
print("human-edit (wiki, 1) pairs:", measure(sc, "(c) materialise human-edit pairs",
                                           lambda: human_p.count()))

c_rbk = measure(sc, "(c) reduceByKey",
                lambda: human_p.reduceByKey(add).takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))
c_gbk = measure(sc, "(c) groupByKey",
                lambda: human_p.groupByKey().mapValues(sum)
                              .takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

assert c_rbk == c_gbk, "the two variants disagree — that would be a real bug"
print("both variants returned the same top-10:", c_rbk == c_gbk)
human_p.unpersist()

cmp = [r for r in RECORDS if r["label"].startswith("(c)") and "materialise" not in r["label"]]
table([{"variant": r["label"].replace("(c) ", ""), "seconds": r["seconds"],
        "tasks": r["tasks"], "shuffle write": fmt_bytes(r["shuffle_write"]),
        "shuffle read": fmt_bytes(r["shuffle_read"])}
       for r in cmp], ["variant", "seconds", "tasks", "shuffle write", "shuffle read"],
      title="(c) reduceByKey vs groupByKey")
if len(cmp) == 2:
    w_ratio = cmp[1]["shuffle_write"] / max(cmp[0]["shuffle_write"], 1)
    print(f"shuffle write ratio groupByKey/reduceByKey = {w_ratio:.2f}x")

human-edit (wiki, 1) pairs: 20923
both variants returned the same top-10: True
(c) reduceByKey vs groupByKey
    variant  seconds   tasks  shuffle write  shuffle read
---------------------------------------------------------
reduceByKey    0.984      10        3.6 KiB       3.6 KiB
 groupByKey    0.838      10        4.1 KiB       4.1 KiB
shuffle write ratio groupByKey/reduceByKey = 1.14x


## (d) Lineage, jobs/stages/tasks per action, and the effect of `cache()`

* `toDebugString()` prints the RDD lineage — the plan Spark would replay for every action,
  which is exactly why an uncached RDD re-reads Kafka on each action.
* The jobs/stages/tasks table comes from the driver's REST API (`/jobs`, `/stages`), the
  same numbers the Spark UI displays, captured per action by `measure()`.
* `cache()` keeps the parsed partitions in memory (1g executor → spills to disk), so the
  second run of the same three actions skips the Kafka re-read *and* the re-parse. The
  accumulator column proves it: it grows once per **recomputed** partition and then stops.


In [7]:
print("=== toDebugString: lineage of the parsed RDD ===")
print(parsed.toDebugString().decode() if isinstance(parsed.toDebugString(), bytes)
      else parsed.toDebugString())

=== toDebugString: lineage of the parsed RDD ===
(6) PythonRDD[43] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[10] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[9] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  SQLExecutionRDD[8] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[7] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[6] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[5] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  SQLExecutionRDD[4] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[3] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  MapPartitionsRDD[2] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  kafka MapPartitionsRDD[1] at javaToPython at NativeMethodAccessorImpl.java:0 []
 |  KafkaSourceRDD[0] at javaToPython at NativeMethodAccessorImpl.java:0 []


In [8]:
seen_before_cache = records_seen.value
print(f"records parsed so far (acc. before cache): {seen_before_cache:,}")
print(f"parse failures so far (acc.)             : {parse_failures.value}")

parsed.persist()
print("cached records:", measure(sc, "(d) cache() + count (materialise)", lambda: parsed.count()))
print(f"records parsed so far (acc. after cache) : {records_seen.value:,} "
      f"-> +{records_seen.value - seen_before_cache:,} for the materialisation")

records parsed so far (acc. before cache): 404,446
parse failures so far (acc.)             : 0
cached records: 57778
records parsed so far (acc. after cache) : 462,224 -> +57,778 for the materialisation


In [9]:
uncached = [r for r in RECORDS if r["label"].startswith("(b-")][:3]
cached = [
    measure(sc, "(d) cached b-i", lambda: top_wikis_by_human_edits(parsed)),
    measure(sc, "(d) cached b-ii", lambda: top_pages_idwiki(parsed)),
    measure(sc, "(d) cached b-iii", lambda: top_words_idwiki_comments(parsed)),
]
assert [b1, b2, b3] == cached, "cached run changed the result — that cannot happen"

pairs = [("b-i top wikis", uncached[0], RECORDS[-3]),
         ("b-ii top pages", uncached[1], RECORDS[-2]),
         ("b-iii top words", uncached[2], RECORDS[-1])]
rows = []
for name, before, after in pairs:
    rows.append({"action": name,
                 "uncached s": before["seconds"], "cached s": after["seconds"],
                 "speedup": round(before["seconds"] / max(after["seconds"], 1e-6), 1)})
table(rows, ["action", "uncached s", "cached s", "speedup"],
      title="(d) cache() impact on the same three actions")

print()
table([{"#": i + 1, "action": r["label"], "jobs": r["jobs"], "stages": r["stages"],
        "tasks": r["tasks"], "shuffle write": fmt_bytes(r["shuffle_write"]),
        "seconds": r["seconds"]} for i, r in enumerate(RECORDS)],
      ["#", "action", "jobs", "stages", "tasks", "shuffle write", "seconds"],
      title="(d) jobs / stages / tasks per action (Spark REST API)")
print()
print(f"records parsed in total: {records_seen.value:,} for a {total:,}-record snapshot — "
      f"the multiplier is the number of recomputations. After the cache it stops growing, "
      f"which is the proof that cache() replaced the recomputation.")

26/11/16 14:08:54 WARN TaskSetManager: Lost task 3.0 in stage 18.0 (TID 101) (10.8.0.6 executor 2): FetchFailed(BlockManagerId(0, 0.0.0.0, 43239, None), shuffleId=5, mapIndex=1, mapId=94, reduceId=3, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:426)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1258)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:988)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:86)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$10.nextCur(Iterator.scala:604)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:618)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.util.CompletionIterator.hasNext(

(d) cache() impact on the same three actions
         action  uncached s  cached s  speedup
----------------------------------------------
  b-i top wikis       3.063    52.526      0.1
 b-ii top pages       6.877    80.302      0.1
b-iii top words        6.89     1.741      4.0

(d) jobs / stages / tasks per action (Spark REST API)
     #                             action    jobs  stages   tasks  shuffle write  seconds
-----------------------------------------------------------------------------------------
     1       snapshot count (whole topic)       1       1       6          0.0 B    4.794
     2                  (a) parse + count       1       1       6          0.0 B    9.109
     3       (b-i) top wikis, human edits       1       2      10        3.6 KiB    3.063
     4            (b-ii) top pages idwiki       1       2      10        1.7 KiB    6.877
     5  (b-iii) top words idwiki comments       1       2      10        2.7 KiB     6.89
     6   (c) materialise human-edit

## Save the results for the worker notebook

The worker notebook re-runs the same pipeline in standalone mode and compares its output
against this file — identical lists mean the distributed run was correct.


In [10]:
SHARED_DIR.mkdir(parents=True, exist_ok=True)
out = {
    "app_id": sc.applicationId,
    "master": sc.master,
    "topic": TOPIC,
    "messages": total,
    "parse_failures": parse_failures.value,
    "partitions": n_partitions,
    "top_wikis": [[w, c] for w, c in b1],
    "top_pages_idwiki": [[t, c] for t, c in b2],
    "top_words_idwiki": [[w, c] for w, c in b3],
    "records": RECORDS,
    "executors": [{"id": e["id"], "hostPort": e["hostPort"], "tasks": e["totalTasks"]}
                  for e in ui_json(sc, "/executors")],
}
path = SHARED_DIR / "master_results.json"
path.write_text(json.dumps(out, indent=1))
print("written:", path)
print("hand this file to q2_spark_worker.ipynb — it compares its standalone run to it.")

written: /tmp/spark-events/q2/master_results.json
hand this file to q2_spark_worker.ipynb — it compares its standalone run to it.


In [11]:
spark.stop()
print("Session stop")

Session stop
